
# Практична робота №1 — Множинна лінійна регресія методом градієнтного спуску

**Предмет:** Машинне навчання  
**Мета:** Реалізувати алгоритм множинної лінійної регресії з нуля (градієнтний спуск), оцінити якість моделі та перевірити **6 класичних припущень МНК**.


## 1. Імпорт бібліотек та базові налаштування

In [ ]:

# Імпорт
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Математика/статистика
try:
    import scipy.stats as stats
except Exception:
    import sys
    !{sys.executable} -m pip -q install scipy
    import scipy.stats as stats

# Діагностика (VIF, DW, Cook's D)
try:
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.stats.stattools import durbin_watson
except Exception:
    import sys
    !{sys.executable} -m pip -q install statsmodels
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from statsmodels.stats.stattools import durbin_watson

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (7, 5)
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## 2. Завантаження даних (EDA)

In [ ]:

# Спроба завантажити реальні дані
POSSIBLE_PATHS = [
    "Student_Performance.csv",
    "/content/Student_Performance.csv",
    "/mnt/data/Student_Performance.csv",
]

csv_path = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is not None:
    df = pd.read_csv(csv_path)
    source = f"Завантажено реальний датасет: {csv_path}"
else:
    # Генерація синтетичних даних, якщо файл відсутній
    rng = np.random.default_rng(RANDOM_STATE)
    n = 200
    Hours_Studied = rng.normal(5, 2, n).clip(0)
    Previous_Scores = rng.normal(70, 10, n).clip(0, 100)
    Extracurricular = rng.integers(0, 2, n)  # 0/1
    Sleep_Hours = rng.normal(7, 1.2, n).clip(3, 10)
    Sample_Question_Papers_Solved = rng.normal(15, 5, n).clip(0)
    Attendance = rng.normal(80, 10, n).clip(40, 100)

    noise = rng.normal(0, 5, n)
    Performance_Index = (
        2.0
        + 1.5 * Hours_Studied
        + 0.4 * Previous_Scores
        + 1.0 * Extracurricular
        + 0.8 * Sleep_Hours
        + 0.3 * Sample_Question_Papers_Solved
        + 0.2 * Attendance
        + noise
    )

    df = pd.DataFrame({
        "Hours_Studied": Hours_Studied,
        "Previous_Scores": Previous_Scores,
        "Extracurricular": Extracurricular,
        "Sleep_Hours": Sleep_Hours,
        "Sample_Question_Papers_Solved": Sample_Question_Papers_Solved,
        "Attendance": Attendance,
        "Performance Index": Performance_Index
    })
    source = "⚠️ Реальний CSV не знайдено — згенеровано синтетичні дані. Замініть на Student_Performance.csv."

print(source)
df.head()


In [ ]:

# Огляд структури
display(df.info())
display(df.describe(numeric_only=True).T)

# Перевірка пропусків
print("\nПропущені значення по стовпцях:")
print(df.isnull().sum())


## 3. Попередня обробка: вибір ознак, train/test, масштабування

In [ ]:

# 3.1. Обробка категоріальних 'Yes'/'No' у числові 0/1
df_proc = df.copy()
for col in df_proc.columns:
    if df_proc[col].dtype == 'object':
        if set(df_proc[col].dropna().unique()) <= {"Yes", "No"}:
            df_proc[col] = df_proc[col].map({"No": 0, "Yes": 1})

# 3.2. Вибір цілі та ознак
target_col = "Performance Index"
feature_cols = [c for c in df_proc.columns if c != target_col]

X = df_proc[feature_cols].values.astype(float)
y = df_proc[target_col].values.astype(float)

# 3.3. train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# 3.4. Масштабування ознак
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 3.5. Додаємо інтерсепт
X_train_final = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_final  = np.c_[np.ones(X_test_scaled.shape[0]),  X_test_scaled]

X_train_final[:3], y_train[:3]


## 4. Градієнтний спуск: реалізація з нуля

In [ ]:

def compute_cost(X, y, b):
    pred = X @ b
    errors = pred - y
    return (errors @ errors) / len(y)

def gradient_descent(X, y, b, learning_rate=0.01, epochs=1000):
    cost_history = []
    m = len(y)
    for _ in range(epochs):
        pred = X @ b
        errors = pred - y
        grad = (2 / m) * (X.T @ errors)
        b = b - learning_rate * grad
        cost_history.append(compute_cost(X, y, b))
    return b, np.array(cost_history)

# Навчання
np.random.seed(RANDOM_STATE)
b0 = np.zeros(X_train_final.shape[1])
learning_rate = 0.05
epochs = 2000

b_final, cost_hist = gradient_descent(X_train_final, y_train, b0, learning_rate, epochs)
b_final, cost_hist[:5]


In [ ]:

# Збіжність
plt.figure()
plt.plot(np.arange(1, len(cost_hist)+1), cost_hist)
plt.xlabel("Ітерація")
plt.ylabel("MSE (train)")
plt.title("Збіжність функції втрат (градієнтний спуск)")
plt.grid(True)
plt.show()

print("Остаточна вартість (train MSE):", float(cost_hist[-1]))


## 5. Оцінка якості на тесті

In [ ]:

y_pred = X_test_final @ b_final

mse_test = np.mean((y_test - y_pred)**2)
r2_test = 1 - (np.sum((y_test - y_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

baseline_pred = np.full_like(y_test, fill_value=y_train.mean())
mse_baseline = np.mean((y_test - baseline_pred)**2)
r2_baseline = 1 - (np.sum((y_test - baseline_pred)**2) / np.sum((y_test - np.mean(y_test))**2))

print(f"MSE (test): {mse_test:.4f}")
print(f"R^2 (test): {r2_test:.4f}")
print(f"Baseline MSE: {mse_baseline:.4f} | Baseline R^2: {r2_baseline:.4f}")

print("\nКоефіцієнти (b_final):")
print(pd.Series(b_final, index=["intercept"] + feature_cols))


## 6. Перевірка 6 припущень лінійної регресії

### 6.1 Лінійність та гомоскедастичність (залишки vs прогнози)

In [ ]:

residuals = y_test - y_pred

plt.figure()
plt.scatter(y_pred, residuals, s=18)
plt.axhline(0, linestyle="--")
plt.xlabel("Прогнозовані значення")
plt.ylabel("Залишки")
plt.title("Residuals vs Predicted")
plt.grid(True)
plt.show()


### 6.2 Нормальність залишків (Q–Q plot + тести)

In [ ]:

plt.figure()
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q–Q plot залишків")
plt.grid(True)
plt.show()

# p-value > 0.05 => не відхиляємо нормальність
if len(residuals) <= 5000:
    shapiro_stat, shapiro_p = stats.shapiro(residuals)
else:
    shapiro_p = np.nan
jb_stat, jb_p = stats.jarque_bera(residuals)

print(f"Shapiro-Wilk p-value: {shapiro_p if not np.isnan(shapiro_p) else 'NA (n>5000)'}")
print(f"Jarque–Bera p-value : {jb_p:.4g}")


### 6.3 Відсутність автокореляції (Durbin–Watson)

In [ ]:

dw = durbin_watson(residuals)
print(f"Durbin–Watson: {dw:.3f}  (~2 — відсутня автокореляція)")


### 6.4 Мультиколінеарність (VIF)

In [ ]:

X_vif = sm.add_constant(X_train_scaled, prepend=True)
vif_vals = []
for i in range(1, X_vif.shape[1]):  # пропускаємо const
    vif_vals.append(variance_inflation_factor(X_vif, i))
vif_df = pd.DataFrame({"feature": feature_cols, "VIF": vif_vals})
vif_df


### 6.5 Викиди та впливові спостереження (Cook's D)

In [ ]:

X_test_ols = sm.add_constant(X_test_scaled, prepend=True)
ols_model = sm.OLS(y_test, X_test_ols).fit()
influence = ols_model.get_influence()
cooks_d = influence.cooks_distance[0]

n_test = len(y_test)
threshold = 4 / n_test
influential_idx = np.where(cooks_d > threshold)[0]

print(f"Поріг Cook's D: {threshold:.4f}")
print(f"К-ть впливових спостережень (Cook's D > 4/n): {len(influential_idx)}")

pd.DataFrame({
    "idx": np.arange(n_test),
    "cooks_d": cooks_d,
    "influential": cooks_d > threshold
}).head(10)


## 7. Висновки


- **Модель** навчена методом градієнтного спуску; наведені коефіцієнти `b_final`.
- Порівняно **MSE** та **R²** з простим бейзлайном.
- Виконано повну **діагностику припущень** (лінійність/гомоскедастичність, нормальність, DW, **VIF**, **Cook’s D**).
- Якщо припущення порушені — опишіть це у звіті та запропонуйте кроки поліпшення: перетворення змінних/цілі, відбір ознак, регуляризація, нелінійні моделі тощо.

> Для захисту використовуйте **реальний** `Student_Performance.csv`.
